# Phase 4: bulk embedding on Colab

Thin wrapper: clones the repo, installs the `embed` extra, builds the
corpus, runs `trustlayer.embed.run` with `--skip-load`, saves each
chunk file to Drive as it lands. No logic lives here — the module owns
everything. Interrupt and re-run any time: completed chunks restore
from Drive and are skipped (resume).

Use a GPU runtime (T4): Runtime → Change runtime type → T4 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/ctru0009/trustlayer.git /content/trustlayer
%cd /content/trustlayer/python
!pip install -q uv
!uv sync --extra embed --extra spark

In [ ]:
# Spark 4.x needs Java 17+; Colab does not reliably ship one.
!apt-get install -qy openjdk-21-jdk-headless > /dev/null 2>&1
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'
!java -version

In [ ]:
# Colab starts with no data: download AESLC + build the corpus (~2 min).
!uv run --extra spark python ../data/scripts/download_aeslc.py
!uv run --extra spark python -m trustlayer.prep.run

In [ ]:
import shutil
import subprocess
from pathlib import Path

OUT = Path('/content/trustlayer/data/processed/embeddings')
BACKUP = Path('/content/drive/MyDrive/trustlayer/embeddings')
OUT.mkdir(parents=True, exist_ok=True)
BACKUP.mkdir(parents=True, exist_ok=True)

# Restore previously saved chunks so the run resumes where it stopped.
for saved in BACKUP.glob('chunk-*.jsonl'):
    if not (OUT / saved.name).exists():
        shutil.copy(saved, OUT / saved.name)
        print(f'[drive] restored {saved.name}')

proc = subprocess.Popen(
    ['uv', 'run', '--extra', 'embed', '--extra', 'spark',
     'python', '-m', 'trustlayer.embed.run', '--skip-load',
     '--device', 'cuda'],
    cwd='/content/trustlayer/python', text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
seen = {p.name for p in BACKUP.glob('chunk-*.jsonl')}
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end='')
    for fresh in OUT.glob('chunk-*.jsonl'):
        if fresh.name not in seen:
            shutil.copy(fresh, BACKUP / fresh.name)
            seen.add(fresh.name)
            print(f'[drive] saved {fresh.name}')
print('exit:', proc.wait())